# Silver — GDP Indicators
Spark SQL. Parses the quarter, dates it to the quarter's first day and keeps the latest version of each quarter.

**Source:** `bronze.statistics_iceland_raw` (`dataset = 'gdp'`)  
**Output:** `silver.gdp_indicators` (Delta), one row per quarter  
**Date rule:** Q1 → Jan 1, Q2 → Apr 1, Q3 → Jul 1, Q4 → Oct 1


In [ ]:
-- Target table (created on the first run)
CREATE TABLE IF NOT EXISTS silver.gdp_indicators (
    quarter       STRING,
    year          INT,
    q             INT,
    date          DATE,
    metric        STRING,
    value         DOUBLE,
    processed_at  TIMESTAMP
) USING DELTA

In [ ]:
-- Upsert: the latest ingested version of each quarter
MERGE INTO silver.gdp_indicators AS target
USING (
    WITH transformed AS (
        SELECT
            period_raw                                AS quarter,
            CAST(SUBSTRING(period_raw, 1, 4) AS INT)  AS year,
            CAST(SUBSTRING(period_raw, 6, 1) AS INT)  AS q,
            CAST(value_raw AS DOUBLE)                 AS value,
            MAKE_DATE(CAST(SUBSTRING(period_raw, 1, 4) AS INT),
                      (CAST(SUBSTRING(period_raw, 6, 1) AS INT) - 1) * 3 + 1, 1) AS date,
            ingested_at
        FROM bronze.statistics_iceland_raw
        WHERE dataset = 'gdp'
          AND value_raw IS NOT NULL
    ),
    latest AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY year, q ORDER BY ingested_at DESC) AS rn
        FROM transformed
        WHERE value IS NOT NULL
    )
    SELECT quarter, year, q, date, 'gdp_yoy_growth' AS metric, value, CURRENT_TIMESTAMP() AS processed_at
    FROM latest
    WHERE rn = 1
) AS source
ON target.year = source.year AND target.q = source.q AND target.metric = source.metric
WHEN MATCHED AND target.value <> source.value THEN UPDATE SET
    target.quarter      = source.quarter,
    target.date         = source.date,
    target.value        = source.value,
    target.processed_at = source.processed_at
WHEN NOT MATCHED THEN INSERT (quarter, year, q, date, metric, value, processed_at)
    VALUES (source.quarter, source.year, source.q, source.date, source.metric, source.value, source.processed_at)

In [ ]:
-- Check
SELECT COUNT(*) AS quarters, MIN(date) AS first_quarter, MAX(date) AS last_quarter
FROM silver.gdp_indicators